# Phase B — Statistical analysis

Four formal tests driving Figure 5:

1. **T1 Species divergence.** Paired Wilcoxon on log2FC[gam] vs log2FC[col] per gene family (ABC, OBP, P450, GST, COE). Tests whether sympatric gambiae and coluzzii mount divergent family-level responses to the same insecticide.

2. **T2 Sweep enrichment.** Permutation null (10,000 random gene sets matched on chromosome-arm distribution). One-sided empirical p + Fisher OR (95% CI) for sweep overlap among Bouaké-sig-up ABCs vs OBPs vs full background. Tests whether efflux upregulation sits on loci already under positive selection in wild West African populations.

3. **T3 OBP genomic clustering.** Permutation test: shuffle DE status among all OBPs; compare observed mean pairwise genomic distance vs null. Tests whether the coluzzii OBP response is driven by a cis-cluster (→ trans-acting regulator) rather than distributed.

4. **T4 Cross-study rank shift.** Mann–Whitney on AnoExpress meta log2FC for Bouaké-sig-up vs Bouaké-ns, within each family. Tests replication in the broader pyrethroid-dominated literature.

All p-values Benjamini–Hochberg corrected across families per test. Effect sizes with 10,000-bootstrap 95% CIs.

In [4]:
import os
os.chdir("/Users/sanjay.nagi/Data-Science/bouake")

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import anoexpress as ae

RNG = np.random.default_rng(42)
N_PERM = 10_000

# ── Data ────────────────────────────────────────────────────────────────
DE_PATH = "results/BouakePM_diffexp.xlsx"
gam = pd.read_excel(DE_PATH, sheet_name="gambiaeCont_gambiaePM")
col = pd.read_excel(DE_PATH, sheet_name="coluzziiCont_coluzziiPM")

# AnoExpress annotations → PFAM/GO lookup per gene
ann = ae.load_annotations()
ann_by_gene = (ann.groupby("gene_id")
               .agg(pfam=("pfamid", lambda x: ";".join(sorted(set(str(v) for v in x if pd.notna(v))))),
                    go=("GO_terms",  lambda x: ";".join(sorted(set(t.strip() for v in x if pd.notna(v) for t in str(v).split(";") if t.strip())))),
                    )
               .reset_index())

# ── Family definitions (PFAM ∪ GO ∪ name regex) ─────────────────────────
FAMILIES = {
    "ABC":  dict(pfam=["PF00005","PF09822","PF12848","PF01061","PF12698"],
                 go=["GO:0042626"], name_regex=r"^Abc"),
    "OBP":  dict(pfam=["PF01395"],
                 go=["GO:0005549"], name_regex=r"obp"),
    "P450": dict(pfam=["PF00067"],
                 go=["GO:0020037","GO:0016705"], name_regex=r"Cyp[0-9]"),
    "GST":  dict(pfam=["PF00043","PF02798","PF13409","PF13410","PF14497"],
                 go=["GO:0004364"], name_regex=r"^Gst"),
    "COE":  dict(pfam=["PF00135"],
                 go=["GO:0052689","GO:0004091"], name_regex=r"^Coe"),
}

def family_ids(defs):
    has_pfam = ann_by_gene["pfam"].apply(lambda s: any(p in s for p in defs["pfam"]))
    has_go   = ann_by_gene["go"].apply(lambda s: any(g in s for g in defs["go"]))
    return set(ann_by_gene.loc[has_pfam | has_go, "gene_id"])

FAM = {name: family_ids(defs) for name, defs in FAMILIES.items()}
for name, ids in FAM.items():
    print(f"{name}: {len(ids)} genes")

ABC: 106 genes
OBP: 46 genes
P450: 190 genes
GST: 38 genes
COE: 89 genes


In [6]:
# ── T1 Species divergence test ──────────────────────────────────────────
# Three linked statistics per family:
#   (i)   Paired Wilcoxon on signed log2FC — test whether distribution differs between species.
#   (ii)  Pearson r (cross-species response concordance) with bootstrap 95% CI.
#   (iii) Sign-concordance rate among sig-DE genes (either species padj<0.05):
#         fraction where log2FC agrees in sign between species.

paired = (gam[["GeneID","log2FoldChange","padj"]]
          .rename(columns={"log2FoldChange":"l2fc_gam","padj":"padj_gam"})
          .merge(col[["GeneID","log2FoldChange","padj"]]
                 .rename(columns={"log2FoldChange":"l2fc_col","padj":"padj_col"}),
                 on="GeneID", how="inner")
          .dropna(subset=["l2fc_gam","l2fc_col"]))

def boot_ci(n, stat_fn, n_boot=10_000, alpha=0.05):
    rng = np.random.default_rng(42)
    idx = rng.integers(0, n, size=(n_boot, n))
    return np.quantile([stat_fn(i) for i in idx], [alpha/2, 1-alpha/2])

def pearson_boot_ci(x, y, n_boot=10_000, alpha=0.05):
    x, y = np.asarray(x), np.asarray(y)
    rng = np.random.default_rng(42)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    rs = np.array([np.corrcoef(x[i], y[i])[0,1] for i in idx])
    return np.quantile(rs, [alpha/2, 1-alpha/2])

rows = []
for fam, ids in FAM.items():
    sub = paired[paired["GeneID"].isin(ids)].copy()
    if len(sub) < 5: continue
    # (i) signed Wilcoxon
    w = stats.wilcoxon(sub["l2fc_gam"], sub["l2fc_col"], alternative="two-sided")
    # (ii) Pearson + 95% bootstrap CI
    r, pr = stats.pearsonr(sub["l2fc_gam"], sub["l2fc_col"])
    r_lo, r_hi = pearson_boot_ci(sub["l2fc_gam"].to_numpy(), sub["l2fc_col"].to_numpy())
    # (iii) sign concordance among sig genes
    sig = sub[(sub["padj_gam"]<0.05) | (sub["padj_col"]<0.05)]
    conc = ((np.sign(sig["l2fc_gam"]) == np.sign(sig["l2fc_col"])).sum() / max(len(sig),1))
    # binomial exact p vs 0.5
    sign_p = stats.binomtest(int((np.sign(sig["l2fc_gam"]) == np.sign(sig["l2fc_col"])).sum()),
                              len(sig), p=0.5).pvalue if len(sig) > 0 else np.nan
    rows.append(dict(family=fam, n_total=len(sub), n_sig=len(sig),
                     med_l2fc_gam=np.median(sub["l2fc_gam"]),
                     med_l2fc_col=np.median(sub["l2fc_col"]),
                     signed_wilcoxon_W=w.statistic, signed_wilcoxon_p=w.pvalue,
                     pearson_r=r, pearson_ci_low=r_lo, pearson_ci_high=r_hi,
                     sign_concordance=conc, sign_binom_p=sign_p))

t1 = pd.DataFrame(rows)
t1["wilcoxon_padj"] = multipletests(t1["signed_wilcoxon_p"], method="fdr_bh")[1]
t1["signconc_padj"] = multipletests(t1["sign_binom_p"], method="fdr_bh")[1]
t1.round(4)

,family,n_total,n_sig,med_l2fc_gam,med_l2fc_col,signed_wilcoxon_W,signed_wilcoxon_p,pearson_r,pearson_ci_low,pearson_ci_high,sign_concordance,sign_binom_p,wilcoxon_padj,signconc_padj
0,ABC,98,55,0.2187,-0.0999,1163.0,0.0000,0.0845,-0.1193,0.2477,0.3818,0.1048,0.0000,0.1746
1,OBP,45,16,0.2706,0.3856,354.0,0.0654,0.0619,-0.2406,0.4605,0.9375,0.0005,0.1091,0.0026
2,P450,179,46,-0.0230,-0.0978,5901.0,0.0027,-0.0996,-0.3278,0.1432,0.3478,0.0541,0.0068,0.1352
3,GST,38,8,0.0958,0.0407,348.0,0.7524,0.1150,-0.3683,0.4000,0.3750,0.7266,0.7524,0.7266
4,COE,89,33,0.0495,0.0452,1800.0,0.4074,0.0077,-0.2808,0.2857,0.4242,0.4869,0.5092,0.6086


In [7]:
# ── T2 Sweep enrichment (chromosome-arm matched permutation) ────────────
# H0: Bouaké-sig-up genes in a family have the same rate of sweep overlap
#     as random gene sets drawn from the same chromosome-arm distribution.
# Test: 10,000 permutations; draw a matched background set, count overlaps.
# Empirical one-sided p = P(null overlap ≥ observed).
# Effect: Fisher OR + 95% CI vs rest of genome.

import malariagen_data
ag3 = malariagen_data.Ag3(results_cache="~/ag3_cache", pre=True)
gs = ag3.geneset()
genes_df = (gs[gs["type"]=="gene"][["contig","start","end","ID","Name"]]
            .rename(columns={"ID":"GeneID"}))

# Convert to atlas compound coords
OFFSETS = {"2R":0, "2L":61545105, "3R":0, "3L":53200684, "X":0}
ARM_TO_CONTIG = {"2R":"2RL","2L":"2RL","3R":"3RL","3L":"3RL","X":"X"}
genes_df = genes_df.assign(
    atlas_contig=genes_df["contig"].map(ARM_TO_CONTIG),
    atlas_start =genes_df["start"] + genes_df["contig"].map(OFFSETS),
    atlas_end   =genes_df["end"]   + genes_df["contig"].map(OFFSETS),
).dropna(subset=["atlas_contig"])

signals = pd.read_csv("skills/selection-atlas/h12-signal-detection-all.csv")
signals["taxon"] = signals["cohort_id"].str.extract(r"_(gamb|colu|arab|bissau|fun)_")
signals["country"] = signals["cohort_id"].str[:2]
WA = {"CI","BF","GH","ML","TG","BJ","GN","GW","GM","SN","SL","LR","NG","NE","CM"}

def gene_overlaps_sweep(gene_row, signals_filtered):
    s = signals_filtered[signals_filtered["contig"] == gene_row["atlas_contig"]]
    if s.empty: return False
    return bool(((s["span1_pstart"] <= gene_row["atlas_end"]) &
                 (s["span1_pstop"]  >= gene_row["atlas_start"])).any())

# Precompute gene→sweep-overlap boolean for gamb+colu West Africa
sig_wa = signals[(signals["taxon"].isin(["gamb","colu"])) & (signals["country"].isin(WA))]
print("sweeps used:", len(sig_wa), "(gamb/colu, West Africa)")

overlap_flag = {}
for _, r in genes_df.iterrows():
    overlap_flag[r["GeneID"]] = gene_overlaps_sweep(r, sig_wa)
overlap_s = pd.Series(overlap_flag, name="sweep_overlap")
print("background sweep-overlap rate: {:.3f} ({}/{})".format(
    overlap_s.mean(), overlap_s.sum(), len(overlap_s)))

I0423 21:32:32.504262 1154437 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.09)

Load genome features: ⠹ (0:00:00.17)

Load genome features: ⠸ (0:00:00.26)

...

Load genome features: ⠦ (0:00:01.39)

Load genome features: ⠧ (0:00:01.55)

Load genome features: ⠇ (0:00:01.65)

sweeps used: 288 (gamb/colu, West Africa)


background sweep-overlap rate: 0.162 (2036/12563)


In [9]:
# Arm-matched permutation (vectorised — 10k perms, fast)
genes_df = genes_df.assign(sweep_overlap=genes_df["GeneID"].map(overlap_flag).fillna(False).astype(bool))

# Precompute per-arm sweep boolean arrays for fast sampling
arm_sweep = {arm: genes_df.loc[genes_df["contig"]==arm, "sweep_overlap"].to_numpy()
             for arm in genes_df["contig"].unique()}
arm_ids = {arm: genes_df.loc[genes_df["contig"]==arm, "GeneID"].to_numpy()
           for arm in genes_df["contig"].unique()}
print({k: len(v) for k,v in arm_sweep.items()})

def perm_enrichment(obs_gene_ids, n_perm=10_000, seed=42):
    obs_set = set(obs_gene_ids) & set(genes_df["GeneID"])
    if not obs_set: return None
    obs_df = genes_df[genes_df["GeneID"].isin(obs_set)]
    obs_arm_counts = obs_df["contig"].value_counts().to_dict()
    obs_hits = int(obs_df["sweep_overlap"].sum())
    obs_n = len(obs_df)
    rng = np.random.default_rng(seed)
    null_hits = np.zeros(n_perm, dtype=np.int32)
    for arm, n in obs_arm_counts.items():
        pool = arm_sweep[arm]
        if n == 0 or len(pool) == 0: continue
        # draw n indices per perm (without replacement per perm)
        for i in range(n_perm):
            idx = rng.choice(len(pool), size=min(n, len(pool)), replace=False)
            null_hits[i] += pool[idx].sum()
    emp_p = float((null_hits >= obs_hits).mean())
    # Fisher OR vs whole genome
    from scipy.stats import fisher_exact
    bg_hits = int(genes_df["sweep_overlap"].sum()) - obs_hits
    bg_n = len(genes_df) - obs_n
    table = [[obs_hits, obs_n-obs_hits],[bg_hits, bg_n-bg_hits]]
    OR, fisher_p = fisher_exact(table, alternative="greater")
    import math
    a,b,c,d = obs_hits, obs_n-obs_hits, bg_hits, bg_n-bg_hits
    if min(a,b,c,d) == 0:
        ci_low = ci_high = np.nan
    else:
        se = math.sqrt(1/a+1/b+1/c+1/d)
        ci_low  = math.exp(math.log(OR) - 1.96*se)
        ci_high = math.exp(math.log(OR) + 1.96*se)
    return dict(n=obs_n, n_hits=obs_hits, rate=obs_hits/obs_n,
                null_mean=null_hits.mean(), null_sd=null_hits.std(),
                perm_p=emp_p, OR=OR, OR_ci_low=ci_low, OR_ci_high=ci_high,
                fisher_p=fisher_p)

gam_sig_up = set(gam[(gam["padj"]<0.05) & (gam["log2FoldChange"]>0)]["GeneID"])
col_sig_up = set(col[(col["padj"]<0.05) & (col["log2FoldChange"]>0)]["GeneID"])

sets_to_test = {
    "ABC-sig-up (gambiae)":   gam_sig_up & FAM["ABC"],
    "OBP-sig-up (coluzzii)":  col_sig_up & FAM["OBP"],
    "ABC-sig-up (coluzzii)":  col_sig_up & FAM["ABC"],
    "OBP-sig-up (gambiae)":   gam_sig_up & FAM["OBP"],
    "P450-sig-up (gambiae)":  gam_sig_up & FAM["P450"],
    "P450-sig-up (coluzzii)": col_sig_up & FAM["P450"],
    "GST-sig-up (gambiae)":   gam_sig_up & FAM["GST"],
    "All sig-up (gambiae)":   gam_sig_up,
    "All sig-up (coluzzii)":  col_sig_up,
}

t2_rows = []
for label, ids in sets_to_test.items():
    res = perm_enrichment(ids)
    if res is None: continue
    t2_rows.append(dict(set=label, **res))
t2 = pd.DataFrame(t2_rows)
t2["perm_padj"] = multipletests(t2["perm_p"], method="fdr_bh")[1]
t2.round(4)

{'2L': 2935, '2R': 3668, '3L': 2211, '3R': 2686, 'X': 1063}


,set,n,n_hits,rate,null_mean,null_sd,perm_p,OR,OR_ci_low,OR_ci_high,fisher_p,perm_padj
0,ABC-sig-up (gambiae),30,8,0.2667,4.4914,1.9261,0.0688,1.8836,0.8374,4.2368,0.1000,0.2199
1,OBP-sig-up (coluzzii),11,1,0.0909,1.6572,1.1730,0.8372,0.5168,0.0661,4.0395,0.8571,0.9838
2,ABC-sig-up (coluzzii),6,2,0.3333,0.9230,0.8614,0.2343,2.5868,0.4735,14.1326,0.2521,0.5272
3,OBP-sig-up (gambiae),11,1,0.0909,1.8800,1.2350,0.8745,0.5168,0.0661,4.0395,0.8571,0.9838
4,P450-sig-up (gambiae),16,5,0.3125,2.3477,1.4047,0.0733,2.3535,0.8168,6.7810,0.1031,0.2199
5,P450-sig-up (coluzzii),9,2,0.2222,1.6116,1.1419,0.5004,1.4777,0.3068,7.1187,0.4419,0.8496
6,GST-sig-up (gambiae),4,0,0.0000,0.5713,0.6954,1.0000,0.0000,NaN,NaN,1.0000,1.0000
7,All sig-up (gambiae),1099,178,0.1620,179.4413,11.4934,0.5664,0.9992,0.8447,1.1820,0.5176,0.8496
8,All sig-up (coluzzii),813,148,0.1820,128.7117,9.8715,0.0283,1.1625,0.9664,1.3985,0.0622,0.2199


In [10]:
# ── T3 OBP genomic clustering (coluzzii response) ───────────────────────
# H0: DE-OBP status is independent of OBP genomic position.
# Test stat: mean pairwise distance among DE-OBPs (lower ⇒ more clustered).
# Permute DE status 10,000× across all OBPs on the same chromosome.
# Repeat for DE-ABCs (gambiae) as comparator.

from itertools import combinations

def mean_pairwise_dist(df):
    # df has columns contig, start. Compute mean abs(start_i - start_j) within contig, summed.
    # Cross-arm pairs get a large fixed penalty so they're less likely by definition.
    if len(df) < 2: return np.nan
    dists = []
    for c, sub in df.groupby("contig"):
        positions = sub["start"].to_numpy()
        if len(positions) < 2: continue
        for i, j in combinations(range(len(positions)), 2):
            dists.append(abs(positions[i]-positions[j]))
    if not dists: return np.nan
    return np.mean(dists)

def cluster_test(family_ids, de_ids, species_label, n_perm=10_000, seed=42):
    fam = genes_df[genes_df["GeneID"].isin(family_ids)][["GeneID","contig","start"]].drop_duplicates("GeneID")
    de  = fam[fam["GeneID"].isin(de_ids)]
    obs = mean_pairwise_dist(de)
    if np.isnan(obs): return None
    rng = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        pick = rng.choice(fam["GeneID"].to_numpy(), size=len(de), replace=False)
        null[i] = mean_pairwise_dist(fam[fam["GeneID"].isin(pick)])
    emp_p = float((null <= obs).mean())  # lower = more clustered
    return dict(species=species_label, n_family=len(fam), n_de=len(de),
                obs_mean_pairwise_dist_bp=obs,
                null_mean=np.nanmean(null), null_sd=np.nanstd(null),
                clustering_p=emp_p)

obp_col_de = set(col[(col["padj"]<0.05) & (col["log2FoldChange"]>0)]["GeneID"]) & FAM["OBP"]
obp_gam_de = set(gam[(gam["padj"]<0.05) & (gam["log2FoldChange"]>0)]["GeneID"]) & FAM["OBP"]
abc_gam_de = set(gam[(gam["padj"]<0.05) & (gam["log2FoldChange"]>0)]["GeneID"]) & FAM["ABC"]
abc_col_de = set(col[(col["padj"]<0.05) & (col["log2FoldChange"]>0)]["GeneID"]) & FAM["ABC"]
p450_gam_de= set(gam[(gam["padj"]<0.05) & (gam["log2FoldChange"]>0)]["GeneID"]) & FAM["P450"]

t3 = pd.DataFrame([
    cluster_test(FAM["OBP"], obp_col_de, "coluzzii OBP DE-up"),
    cluster_test(FAM["OBP"], obp_gam_de, "gambiae OBP DE-up"),
    cluster_test(FAM["ABC"], abc_gam_de, "gambiae ABC DE-up"),
    cluster_test(FAM["ABC"], abc_col_de, "coluzzii ABC DE-up"),
    cluster_test(FAM["P450"], p450_gam_de, "gambiae P450 DE-up"),
])
t3["padj"] = multipletests(t3["clustering_p"], method="fdr_bh")[1]
t3.round(4)

,species,n_family,n_de,obs_mean_pairwise_dist_bp,null_mean,null_sd,clustering_p,padj
0,coluzzii OBP DE-up,43,11,1.682847e+07,1.542385e+07,4.320802e+06,0.6218,0.6218
1,gambiae OBP DE-up,43,11,1.570983e+07,1.542385e+07,4.320802e+06,0.5063,0.6218
2,gambiae ABC DE-up,95,30,1.458398e+07,1.719514e+07,2.027980e+06,0.1038,0.2595
3,coluzzii ABC DE-up,95,6,3.155000e+03,1.693439e+07,8.469400e+06,0.0001,0.0005
4,gambiae P450 DE-up,171,16,1.032664e+07,1.304992e+07,3.112483e+06,0.1953,0.3255


In [11]:
# Investigate the tight coluzzii ABC cluster (p<0.0001)
print("coluzzii ABC DE-up genes:")
abc_col_de_df = genes_df[genes_df["GeneID"].isin(abc_col_de)][["GeneID","contig","start","end","Name"]]
abc_col_de_df = abc_col_de_df.merge(col[["GeneID","GeneName","log2FoldChange","padj"]], on="GeneID")
print(abc_col_de_df.sort_values(["contig","start"]).to_string(index=False))

print("\ngambiae ABC DE-up positions (head):")
abc_gam_de_df = genes_df[genes_df["GeneID"].isin(abc_gam_de)][["GeneID","contig","start","end"]]
abc_gam_de_df = abc_gam_de_df.merge(gam[["GeneID","GeneName","log2FoldChange","padj"]], on="GeneID")
print(abc_gam_de_df.sort_values(["contig","start"]).head(10).to_string(index=False))

# OBP col positions too
print("\ncoluzzii OBP DE-up positions:")
obp_col_de_df = genes_df[genes_df["GeneID"].isin(obp_col_de)][["GeneID","contig","start","end"]]
obp_col_de_df = obp_col_de_df.merge(col[["GeneID","GeneName","log2FoldChange","padj"]], on="GeneID")
print(obp_col_de_df.sort_values(["contig","start"]).to_string(index=False))

coluzzii ABC DE-up genes:
    GeneID contig    start      end   Name GeneName  log2FoldChange     padj
AGAP006427     2L 31735666 31752427  ABCC7    ABCC7        0.802251 0.000070
AGAP002638     2R 24825285 24848323  ABCH1    ABCH1        0.723410 0.000059
AGAP011271     3L 20401856 20423070    NaN      NaN        0.414214 0.004998
AGAP009467     3R 34003554 34006145 ABCG16   ABCG16        0.460293 0.038883
AGAP009468     3R 34006709 34009946 ABCG17   ABCG17        0.643068 0.017547
AGAP000553      X  9874701  9885361  white    white        0.540346 0.044806

gambiae ABC DE-up positions (head):
    GeneID contig    start      end GeneName  log2FoldChange         padj
AGAP005134     2L 10355282 10358610      NaN        0.446965 2.440529e-02
AGAP005639     2L 18092053 18120216    ABCB2        0.776401 2.569427e-05
AGAP006379     2L 30925949 30932896    ABCA1        0.931474 2.091206e-04
AGAP006380     2L 30934786 30941227    ABCA2        1.837999 3.065699e-07
AGAP001588     2R  6491877  

In [ ]:
# T3 redux — clustering test using two proper metrics.
# (a) Arm concentration: largest fraction of DE genes on a single arm (vs null from family).
# (b) Median nearest-neighbour distance within same arm (cross-arm neighbours → genome_size penalty).
# Previous "mean within-arm pairwise" was dominated by 2-gene coincidences — dropped.

GENOME_PENALTY = 3e8  # larger than any chromosome, used for missing within-arm neighbours

def nn_dist_statistic(df):
    """Median nearest-neighbour distance (bp). Genes with no within-arm sibling get GENOME_PENALTY."""
    if len(df) < 2: return np.nan
    nn = []
    for _, row in df.iterrows():
        same_arm = df[(df["contig"]==row["contig"]) & (df["GeneID"]!=row["GeneID"])]
        if same_arm.empty:
            nn.append(GENOME_PENALTY)
        else:
            nn.append(int((same_arm["start"] - row["start"]).abs().min()))
    return float(np.median(nn))

def arm_concentration(df):
    """Largest fraction on single arm."""
    return float(df["contig"].value_counts(normalize=True).iloc[0]) if len(df) else np.nan

def cluster_test_v2(family_ids, de_ids, label, n_perm=10_000, seed=42):
    fam = genes_df[genes_df["GeneID"].isin(family_ids)][["GeneID","contig","start"]].drop_duplicates("GeneID")
    de  = fam[fam["GeneID"].isin(de_ids)]
    if len(de) < 3: return None
    obs_nn = nn_dist_statistic(de)
    obs_arm = arm_concentration(de)
    rng = np.random.default_rng(seed)
    null_nn = np.empty(n_perm); null_arm = np.empty(n_perm)
    pool = fam["GeneID"].to_numpy()
    for i in range(n_perm):
        pick = rng.choice(pool, size=len(de), replace=False)
        d = fam[fam["GeneID"].isin(pick)]
        null_nn[i] = nn_dist_statistic(d)
        null_arm[i]= arm_concentration(d)
    # clustering → smaller nn, larger arm_concentration
    p_nn  = float(np.mean(null_nn <= obs_nn))
    p_arm = float(np.mean(null_arm >= obs_arm))
    # arm distribution detail
    arms = de["contig"].value_counts().to_dict()
    return dict(set=label, n_family=len(fam), n_de=len(de),
                obs_nn_bp=obs_nn, null_nn_bp=np.median(null_nn),
                p_nn=p_nn, obs_arm_conc=obs_arm, null_arm_conc=np.median(null_arm),
                p_arm=p_arm, arm_counts=str(arms))

rows = []
for label, fam_ids, de_ids in [
    ("coluzzii OBP-up",   FAM["OBP"], obp_col_de),
    ("gambiae OBP-up",    FAM["OBP"], obp_gam_de),
    ("gambiae ABC-up",    FAM["ABC"], abc_gam_de),
    ("coluzzii ABC-up",   FAM["ABC"], abc_col_de),
    ("gambiae P450-up",   FAM["P450"], p450_gam_de),
]:
    r = cluster_test_v2(fam_ids, de_ids, label)
    if r: rows.append(r)

t3 = pd.DataFrame(rows)
t3["p_nn_padj"]  = multipletests(t3["p_nn"],  method="fdr_bh")[1]
t3["p_arm_padj"] = multipletests(t3["p_arm"], method="fdr_bh")[1]
t3.round(4)